# Introduction to Neural Networks and Deep Learning

## Learning Objectives
- Understand the basic concepts of neural networks
- Learn about forward and backward propagation
- Explore different activation functions
- Implement a simple neural network from scratch

## 1.1 What is a Neural Network?

A neural network is a computational model inspired by biological neural networks. It consists of:
- **Neurons**: Basic processing units
- **Connections**: Links between neurons with weights
- **Layers**: Organized structure of neurons

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 1.2 Single Neuron (Perceptron)

In [ ]:
class Perceptron:
    def __init__(self, input_size):
        self.weights = np.random.randn(input_size)
        self.bias = np.random.randn()
    
    def forward(self, x):
        """Forward pass: compute weighted sum + bias"""
        return np.dot(x, self.weights) + self.bias
    
    def step_function(self, x):
        """Step activation function"""
        return 1 if x >= 0 else 0
    
    def predict(self, x):
        return self.step_function(self.forward(x))

# Example usage
perceptron = Perceptron(input_size=2)
x_test = np.array([1.0, -0.5])
output = perceptron.predict(x_test)
print(f"Input: {x_test}, Output: {output}")

## 1.3 Activation Functions

In [ ]:
def sigmoid(x):
    """Sigmoid activation function"""
    return 1 / (1 + np.exp(-x))

def tanh(x):
    """Hyperbolic tangent activation function"""
    return np.tanh(x)

def relu(x):
    """Rectified Linear Unit activation function"""
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    """Leaky ReLU activation function"""
    return np.where(x > 0, x, alpha * x)

# Visualize activation functions
x = np.linspace(-5, 5, 100)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(x, sigmoid(x))
plt.title('Sigmoid')
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(x, tanh(x))
plt.title('Tanh')
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(x, relu(x))
plt.title('ReLU')
plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(x, leaky_relu(x))
plt.title('Leaky ReLU')
plt.grid(True)

plt.tight_layout()
plt.show()

## 1.4 Multi-Layer Neural Network

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes):
        """Initialize neural network with given layer sizes"""
        self.layer_sizes = layer_sizes
        self.weights = []
        self.biases = []
        
        # Initialize weights and biases
        for i in range(len(layer_sizes) - 1):
            self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.1)
            self.biases.append(np.random.randn(layer_sizes[i+1]) * 0.1)
    
    def forward(self, X):
        """Forward pass through the network"""
        activations = [X]
        z_values = []
        
        for i in range(len(self.weights)):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            z_values.append(z)
            
            if i < len(self.weights) - 1:  # Hidden layers
                activation = relu(z)
            else:  # Output layer
                activation = sigmoid(z)
            
            activations.append(activation)
        
        return activations[-1], activations, z_values
    
    def compute_loss(self, y_true, y_pred):
        """Compute binary cross-entropy loss"""
        m = y_true.shape[0]
        # Avoid division by zero
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = -(1/m) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss

## 1.5 Training the Network

In [ ]:
# Generate sample data
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
y = y.reshape(-1, 1)  # Reshape for compatibility

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualize the data
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), cmap='viridis')
plt.title('Training Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test.ravel(), cmap='viridis')
plt.title('Test Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Initialize neural network
nn = NeuralNetwork(layer_sizes=[2, 10, 10, 1])  # 2 inputs, 2 hidden layers with 10 neurons, 1 output

# Training parameters
learning_rate = 0.01
epochs = 1000

# Training loop
losses = []

for epoch in range(epochs):
    # Forward pass
    y_pred, activations, z_values = nn.forward(X_train)
    
    # Compute loss
    loss = nn.compute_loss(y_train, y_pred)
    losses.append(loss)
    
    # Backward pass (simplified gradient computation)
    m = X_train.shape[0]
    
    # Output layer gradients
    dz3 = y_pred - y_train
    dw3 = (1/m) * np.dot(activations[2].T, dz3)
    db3 = (1/m) * np.sum(dz3, axis=0)
    
    # Second hidden layer gradients
    dz2 = np.dot(dz3, nn.weights[2].T) * (z_values[1] > 0)
    dw2 = (1/m) * np.dot(activations[1].T, dz2)
    db2 = (1/m) * np.sum(dz2, axis=0)
    
    # First hidden layer gradients
    dz1 = np.dot(dz2, nn.weights[1].T) * (z_values[0] > 0)
    dw1 = (1/m) * np.dot(X_train.T, dz1)
    db1 = (1/m) * np.sum(dz1, axis=0)
    
    # Update weights and biases
    nn.weights[2] -= learning_rate * dw3
    nn.biases[2] -= learning_rate * db3
    nn.weights[1] -= learning_rate * dw2
    nn.biases[1] -= learning_rate * db2
    nn.weights[0] -= learning_rate * dw1
    nn.biases[0] -= learning_rate * db1
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 1.6 Model Evaluation

In [ ]:
# Make predictions on test set
y_pred_test, _, _ = nn.forward(X_test)
y_pred_classes = (y_pred_test > 0.5).astype(int)

# Calculate accuracy
accuracy = np.mean(y_pred_classes == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Visualize predictions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test.ravel(), cmap='viridis')
plt.title('True Labels')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 3, 2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_pred_classes.ravel(), cmap='viridis')
plt.title('Predicted Labels')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 3, 3)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_pred_test.ravel(), cmap='viridis')
plt.title('Prediction Probabilities')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar()

plt.tight_layout()
plt.show()

## 1.7 Key Takeaways

- **Neural networks** are composed of interconnected layers of neurons
- **Activation functions** introduce non-linearity, enabling complex function approximation
- **Forward propagation** computes predictions through the network
- **Backward propagation** computes gradients for weight updates
- **Loss functions** measure the difference between predictions and true values

## Exercises

1. Experiment with different activation functions and observe their effects
2. Change the network architecture (number of layers/neurons) and compare performance
3. Implement different loss functions (MSE, etc.)
4. Add momentum to the gradient descent algorithm